# GameTheory-13f : Ataraxos — le patron en trois briques sur Kuhn Poker

**Navigation** : precedent GameTheory-13e-Revision-Croyances-Adversaire (lien entrant pose chez lui) · [Index](README.md)

> **Ce que ce carnet est.** Une distillation pédagogique du *patron de conception* d'**Ataraxos** (Sokota, Vinitsky, Hu, Fan, Kolter, Farina — *Scalable Decision Making for Games of Imperfect Information*, arXiv 2511.07312, v2 2026), exécutée de bout en bout sur un jeu-jouet, Kuhn Poker. Il prolonge la sous-série « autour de 13 » : le CFR de `GameTheory-13` résout par regret, Ataraxos **apprend par auto-jeu puis cherche au temps de test**.

> **Ce que ce carnet n'est pas.** Aucune reproduction de Stratego : l'entraînement du papier a consommé 16 H100 pendant une semaine (~8 000 $ aux prix de 2025, contre ~3-4,5 M$ et 1 024 TPU pendant 2-3 mois pour DeepNash — ces chiffres sont **cités de l'article**, jamais mesurés ici). Aucun claim de surhumanité locale : les mesures de ce carnet portent sur des propriétés du patron à l'échelle d'un jouet.

> **Copie pédagogique déclarée** (règle *organ-first*) : l'organe natif est le code MIT de l'organisation [`AtaraxosAI`](https://github.com/AtaraxosAI) (dépôts `stratego`, `doudizhu`, `hanabi`), invoqué et mesuré dans le **grain B** de #20408. Ici, l'implémentation est volontairement locale et minimale : le but est de rendre chaque brique lisible, pas d'approcher le moteur CUDA du papier. Les 5 questions *organ-first* sont traitées dans le corps de la PR. Le jeu-jouet lui-même reprend les sémantiques de l'environnement Kuhn de `GameTheory-13-ImperfectInfo-CFR-Python` (organe interne de la série, cité, non réimporté — chaque carnet de la série est auto-contenu).

## 1. Le papier en trois briques, et pourquoi ça tient dans un carnet

Ataraxos a battu le champion du monde de Stratego (15 victoires, 1 défaite, 4 nulles ; p < 2,6e-4 sous hypothèse i.i.d.), puis 38-2-0 en démonstration au championnat 2025, et établit un état de l'art à Hanabi (2 à 5 joueurs) et au dou dizhu — tous jeux à information cachée massive (plus de 10^33 configurations au Stratego). **Ces résultats sont ceux du papier : ce carnet les cite, il ne les mesure pas.**

Ce qui est replicable sur un siège, c'est le **patron** — identique pour tous les jeux du papier :

1. **Brique 1 — un réseau politique-valeur** entraîné par auto-jeu : il propose les décisions et prédit l'issue ;
2. **Brique 2 — un réseau de croyances** : entraîné sur les données d'auto-jeu, il modélise l'information cachée (ici : la carte de l'adversaire) et sert de **modèle génératif** pour échantillonner des états cohérents avec ce que le joueur sait ;
3. **Brique 3 — une recherche au temps de test** par *update equivalence* : la recherche échantillonne des états depuis le réseau de croyances, déroule des rollouts à profondeur bornée, moyenne les valeurs prédites, puis applique **un pas tabulaire de descente miroir magnétique — avec exactement les mêmes divergences reverse-KL qu'à l'entraînement**. Chercher, c'est appliquer la mise à jour d'entraînement aux croyances du moment.

Deux mécanismes de stabilisation complètent le patron (l'ablation est l'Exercice 2) :

- ***dynamic damping*** : la force de régularisation et la taille du pas sont **coordonnées** — régularisation forte et pas agressifs tôt dans l'auto-jeu, régularisation faible et petits pas tard. C'est ce qui rend l'auto-jeu stable sans reweighting d'importance des trajectoires ni moyennage de politiques ;
- le pas est contrôlé par quatre mécanismes complémentaires : pénalité reverse-KL vers la politique de collecte, clipping du ratio d'importance, clipping de la norme de gradient, pas d'Adam. Ce carnet implémente la pénalité reverse-KL, le clipping de norme de gradient et le pas d'Adam décroissant ; les deux autres sont cités.

**Plan** : environnement Kuhn (§2), brique 1 (§3), brique 2 (§4), brique 3 et son témoin négatif (§5-6), exercices (§7), raccords (§8).

## 2. Kuhn Poker : le jeu-jouet

Rappels (mêmes conventions que `GameTheory-13`) : 3 cartes J < Q < K, deux joueurs, une ante de 1 chacun, un tour d'enchères de taille 1. Historiques d'actions en lettres : `p` passe, `b` enchérit, `f` se couche, `c` suit. Chaque joueur voit sa carte et l'historique public — jamais celle de l'autre : c'est l'information imparfaite que la brique 2 doit modéliser.

Le carnet encode chaque **état d'information** (infoset) comme un petit vecteur : joueur (2), carte privée (3), position dans le tour (2 — agir à l'ouverture ou répondre à une enchère). Kuhn a exactement **12 infosets** (6 par joueur), tous à **2 actions légales** : c'est ce qui rend chaque brique du patron lisible sans machinerie.

In [1]:
import random

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

SEED = 0
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.set_num_threads(2)

# ---------------------------------------------------------------- Kuhn Poker
# Semantiques identiques a l'environnement du carnet GameTheory-13 (serie).
CARD_NAMES = ['Valet', 'Dame', 'Roi']


class KuhnPoker:
    """Environnement Kuhn a 2 joueurs, 3 cartes, un tour d'encheres de taille 1."""

    @staticmethod
    def deal():
        return tuple(random.sample(range(3), 2))  # (carte J1, carte J2)

    @staticmethod
    def legal_actions(history):
        # Ouverture (historique vide ou 'p') : passer ou encherir.
        # Reponse a une enchere ('b' ou 'pb') : se coucher ou suivre.
        if history in ('', 'p'):
            return ['p', 'b']
        return ['f', 'c']

    @staticmethod
    def current_player(history):
        # J1 ouvre ; J2 parle sur 'p' ou repond sur 'b' ; J1 repond sur 'pb'.
        return 0 if history in ('', 'pb') else 1

    @staticmethod
    def is_terminal(history):
        return history in ('pp', 'bf', 'pbf', 'bc', 'pbc')

    @staticmethod
    def payoff(history, cards):
        """Gain du joueur 1 (le joueur 2 prend l'oppose)."""
        c1, c2 = cards
        winner = 0 if c1 > c2 else 1
        if history == 'pp':
            return 1.0 if winner == 0 else -1.0
        if history == 'pbf':      # J1 passe, J2 encherit, J1 se couche
            return -1.0
        if history == 'bf':       # J1 encherit, J2 se couche
            return 1.0
        # Showdown apres une enchere suivie : pot de 2 par cote
        return 2.0 if winner == 0 else -2.0


def infoset_id(player, card, bucket):
    """Les 12 infosets de Kuhn : (joueur, carte, position). bucket 0 = ouvrir, 1 = repondre."""
    return player * 6 + card * 2 + bucket


def history_bucket(history):
    return 0 if history in ('', 'p') else 1


def history_of_infoset(iset):
    """Historique (pour le joueur courant) correspondant a l infoset."""
    player, card, bucket = iset // 6, (iset % 6) // 2, iset % 2
    if player == 0:
        return 'pb' if bucket == 1 else ''
    return 'b' if bucket == 1 else 'p'


def encode_infoset(iset):
    """Vecteur 7-dim : one-hot joueur (2) + one-hot carte (3) + one-hot position (2)."""
    player, card, bucket = iset // 6, (iset % 6) // 2, iset % 2
    v = np.zeros(7, dtype=np.float32)
    v[player] = 1.0
    v[2 + card] = 1.0
    v[5 + bucket] = 1.0
    return v


ALL_INFOSETS = list(range(12))
ENCODED = torch.tensor(np.stack([encode_infoset(i) for i in ALL_INFOSETS]))

print('Infosets de Kuhn :', len(ALL_INFOSETS))
for h in ('', 'p', 'b', 'pb'):
    pl = KuhnPoker.current_player(h)
    print(f"  historique {h!r:6s} -> joueur {pl + 1}, actions legales {KuhnPoker.legal_actions(h)}")

Infosets de Kuhn : 12
  historique ''     -> joueur 1, actions legales ['p', 'b']
  historique 'p'    -> joueur 2, actions legales ['p', 'b']
  historique 'b'    -> joueur 2, actions legales ['f', 'c']
  historique 'pb'   -> joueur 1, actions legales ['f', 'c']


### Interprétation : l'environnement et ses états d'information

L'environnement ne connaît que l'arbre public : les historiques, les actions légales, le joueur courant et le gain terminal. L'**information cachée** (la carte adverse) n'apparaît nulle part dans l'encodage d'infoset — un joueur ne conditionne sa décision que sur `(joueur, ma carte, position)`. C'est précisément cette absence que le réseau de croyances de la brique 2 viendra combler **en appris** : il prédit la distribution de la carte adverse à partir de ce que le joueur sait.

## 3. Brique 1 — le réseau politique-valeur et l'auto-jeu amorti

Le réseau : un tronc MLP partagé, une tête **politique** (2 logits — une par action légale) et une tête **valeur** (gain espéré depuis l'infoset). L'entraînement est de l'auto-jeu : les deux joueurs partagent le même réseau, les données viennent des parties qu'il joue contre lui-même.

La boucle d'entraînement applique la recette de stabilisation du papier, à l'échelle du jouet :

- **pénalité reverse-KL** vers la *politique de collecte* (un instantané du réseau pris au début de chaque manche) — c'est la divergence qu'utilise aussi la brique 3 ;
- **clipping de la norme de gradient** (1.0) et **pas d'Adam décroissant** ;
- ***dynamic damping*** : le coefficient de pénalité `lambda` et le pas `lr` suivent une **coordination** — forte régularisation et pas agressifs tôt, régularisation faible et petits pas tard.

La fonction objectif est un gradient de politique avec valeur de référence (`avantage = gain réalisé − valeur prédite`), la régression de la tête valeur vers le gain réalisé, et la pénalité reverse-KL **exacte** — exacte parce qu'avec 12 infosets on peut la sommer sur tous les infosets au lieu de l'estimer.

In [2]:
class PolicyValueNet(nn.Module):
    """Tronc partage, tete politique (2 logits) et tete valeur (1 scalaire)."""

    def __init__(self, dim_in=7, hidden=64):
        super().__init__()
        self.trunk = nn.Sequential(nn.Linear(dim_in, hidden), nn.ReLU(),
                                   nn.Linear(hidden, hidden), nn.ReLU())
        self.policy_head = nn.Linear(hidden, 2)
        self.value_head = nn.Linear(hidden, 1)

    def forward(self, x):
        z = self.trunk(x)
        return self.policy_head(z), self.value_head(z).squeeze(-1)

    @torch.no_grad()
    def policy_table(self):
        """Politique comportementale tabulaire : {infoset: [p(action0), p(action1)]}."""
        logits, _ = self(ENCODED)
        probs = F.softmax(logits, dim=-1).numpy()
        return {i: probs[i].tolist() for i in ALL_INFOSETS}


NET = PolicyValueNet()
OPTIM = torch.optim.Adam(NET.parameters(), lr=3e-3)
print(NET)


def play_one_game(policy_fn):
    """Joue une partie complete en echantillonnant depuis policy_fn(iset) -> [p0, p1].
    Rend (gain J1, historique, cartes, [(iset, action_idx, joueur)])."""
    cards = KuhnPoker.deal()
    history = ''
    decisions = []
    while not KuhnPoker.is_terminal(history):
        player = KuhnPoker.current_player(history)
        iset = infoset_id(player, cards[player], history_bucket(history))
        probs = policy_fn(iset)
        a_idx = int(random.choices((0, 1), weights=probs)[0])
        legal = KuhnPoker.legal_actions(history)
        decisions.append((iset, a_idx, player))
        history += legal[a_idx]
    return KuhnPoker.payoff(history, cards), history, cards, decisions


def rollout_return(history, cards, player, policy_fn, max_depth=2):
    """Complete la partie depuis history en echantillonnant policy_fn, rend le gain de player."""
    depth = 0
    while not KuhnPoker.is_terminal(history) and depth < max_depth:
        cur = KuhnPoker.current_player(history)
        iset = infoset_id(cur, cards[cur], history_bucket(history))
        probs = policy_fn(iset)
        a_idx = int(random.choices((0, 1), weights=probs)[0])
        history += KuhnPoker.legal_actions(history)[a_idx]
        depth += 1
    g = KuhnPoker.payoff(history, cards)
    return g if player == 0 else -g


print('Reseau pret. Politique initiale (proche uniforme attendue) sur 3 infosets :')
for i in (0, 1, 6):
    print(f'  infoset {i:2d} -> {[round(p, 4) for p in NET.policy_table()[i]]}')

PolicyValueNet(
  (trunk): Sequential(
    (0): Linear(in_features=7, out_features=64, bias=True)
    (1): ReLU()
    (2): Linear(in_features=64, out_features=64, bias=True)
    (3): ReLU()
  )
  (policy_head): Linear(in_features=64, out_features=2, bias=True)
  (value_head): Linear(in_features=64, out_features=1, bias=True)
)
Reseau pret. Politique initiale (proche uniforme attendue) sur 3 infosets :
  infoset  0 -> [0.508, 0.492]
  infoset  1 -> [0.5074, 0.4926]
  infoset  6 -> [0.5104, 0.4896]


In [3]:
def train_selfplay(net, optim, n_rounds=28, games_per_round=256,
                    lam_start=0.8, lam_end=0.05, lr_start=3e-3, lr_end=2e-4,
                    seed=SEED, verbose=True):
    """Auto-jeu avec dynamic damping : lambda fort et pas agressif tot,
    lambda faible et petits pas tard. Rend l'historique de la penalite KL."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    kl_history = []

    def policy_fn(iset):
        return TAB[iset]

    for r in range(n_rounds):
        frac = r / max(n_rounds - 1, 1)
        lam = lam_start + (lam_end - lam_start) * frac       # fort -> faible
        for g in optim.param_groups:
            g['lr'] = lr_start + (lr_end - lr_start) * frac  # agressif -> petit

        TAB = net.policy_table()  # politique de collecte de la manche

        # ---- collecte des trajectoires d'auto-jeu
        games = [play_one_game(policy_fn) for _ in range(games_per_round)]
        payoffs = np.array([g[0] for g in games])
        round_decisions = [(d[0], d[1], g[0] if d[2] == 0 else -g[0])
                           for g in games for d in g[3]]

        # ---- passe d'optimisation sur les decisions de la manche
        idx = torch.tensor([d[0] for d in round_decisions])
        acts = torch.tensor([d[1] for d in round_decisions])
        rets = torch.tensor([d[2] for d in round_decisions], dtype=torch.float32)
        x = ENCODED[idx]
        logits, values = net(x)
        logp = F.log_softmax(logits, dim=-1)
        chosen_logp = logp[torch.arange(len(acts)), acts]
        with torch.no_grad():
            advantage = rets - values
        pg_loss = -(chosen_logp * advantage).mean()
        v_loss = F.mse_loss(values, rets)

        # reverse-KL exacte vers la politique de collecte (TAB), sur TOUS les infosets
        with torch.no_grad():
            col_logp = torch.log(torch.tensor(
                np.stack([TAB[i] for i in ALL_INFOSETS]),
                dtype=torch.float32) + 1e-12)
        new_logp = F.log_softmax(net(ENCODED)[0], dim=-1)
        kl = ((new_logp.exp() * (new_logp - col_logp)).sum(-1)).mean()

        loss = pg_loss + 0.5 * v_loss + lam * kl
        optim.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(net.parameters(), 1.0)
        optim.step()

        # distance reellement parcourue : KL(apres pas) vers la collecte
        with torch.no_grad():
            post = net(ENCODED)[0]
            post_logp = F.log_softmax(post, dim=-1)
            kl_after = float((post_logp.exp() * (post_logp - col_logp)
                              ).sum(-1).mean())
        kl_history.append(kl_after)
        if verbose and (r % 7 == 0 or r == n_rounds - 1):
            print(f'round {r:2d}: gain moyen J1 = {payoffs.mean():+.4f} | '
                  f'KL(collecte, apres pas) = {kl_after:.4f} | lambda = {lam:.3f} | '
                  f'lr = {optim.param_groups[0]["lr"]:.1e}')
    return kl_history


KL_HIST = train_selfplay(NET, OPTIM)
POLICY_BASE = NET.policy_table()
print('\nPolitique base apres auto-jeu (3 infosets) :')
for i in (0, 1, 6):
    print(f'  infoset {i:2d} -> {[round(p, 4) for p in POLICY_BASE[i]]}')

round  0: gain moyen J1 = +0.0664 | KL(collecte, apres pas) = 0.0008 | lambda = 0.800 | lr = 3.0e-03
round  7: gain moyen J1 = +0.3125 | KL(collecte, apres pas) = 0.0012 | lambda = 0.606 | lr = 2.3e-03
round 14: gain moyen J1 = +0.1367 | KL(collecte, apres pas) = 0.0008 | lambda = 0.411 | lr = 1.5e-03
round 21: gain moyen J1 = -0.1133 | KL(collecte, apres pas) = 0.0002 | lambda = 0.217 | lr = 8.2e-04
round 27: gain moyen J1 = +0.0430 | KL(collecte, apres pas) = 0.0000 | lambda = 0.050 | lr = 2.0e-04

Politique base apres auto-jeu (3 infosets) :
  infoset  0 -> [0.1158, 0.8842]
  infoset  1 -> [0.1336, 0.8664]
  infoset  6 -> [0.1083, 0.8917]


### Interprétation : ce que l'auto-jeu apprend — et ce qu'il n'apprend pas

**À lire sur la sortie ci-dessus.** Le gain moyen du joueur 1 oscille autour de 0 : c'est la signature d'un auto-jeu équilibré — le même réseau joue les deux côtés, aucun des deux n'a d'avantage systématique. La politique qui en sort est **très asymétrique en apparence** : sur les trois infosets affichés — tous tenant un Valet, en ouverture comme en réponse — elle enchérit 86,6–89,2 %, un bluff quasi systématique qu'il faudra garder en tête pour lire les croyances de la brique 2.

**La colonne KL se lit comme la distance réellement parcourue.** La pénalité reverse-KL compare le réseau à la **table de collecte figée en début de manche** (le comparer au réseau lui-même donnerait toujours 0 — l'implémentation le documente). La valeur affichée est la KL **après** le pas d'optimisation : ici ≤ 1.2·10⁻³, et elle tombe vers 0 en fin d'entraînement quand le pas d'apprentissage s'éteint. La région de confiance **borne le pas**, elle ne l'annule pas — un seul pas de gradient sépare le réseau de sa table, et la KL le mesure.

Ce que l'auto-jeu **n'apprend pas** : l'équilibre de Nash. Le CFR du carnet 13 converge vers l'équilibre par construction ; l'auto-jeu par gradient de politique, lui, converge vers une **réponse à sa propre politique** — les régimes d'entraînement auto-jeu s'écartent typiquement de Nash, et c'est précisément le vide que la brique 3 viendra combler au temps de test. L'exploitabilité mesurée en §5 quantifiera cet écart.

## 4. Brique 2 — le réseau de croyances

Le réseau de croyances modélise **l'information cachée** : étant donné ce qu'un joueur sait (sa carte, la position dans le tour), quelle est la distribution de la **carte adverse** ? Il est entraîné **sur les données d'auto-jeu** de la brique 1, en supervisé : pour chaque décision jouée, l'étiquette est la carte adverse réelle — connue du journal d'entraînement, jamais du joueur.

C'est le **modèle génératif** du patron : au temps de test, la recherche n'invente pas des états, elle **échantillonne des mondes cohérents avec l'observation** depuis ce réseau. Le carnet 13e fait le même geste en symbolique (révision AGM des croyances sur l'adversaire) : ici la distribution est **apprise** — pont naturel et témoin négatif réciproque entre les deux satellites.

In [4]:
class BeliefNet(nn.Module):
    """P(carte adverse | mon infoset) : softmax sur 3 cartes."""

    def __init__(self, dim_in=7, hidden=32):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(dim_in, hidden), nn.ReLU(),
                                 nn.Linear(hidden, 3))

    def forward(self, x):
        return self.net(x)


BELIEF = BeliefNet()
B_OPTIM = torch.optim.Adam(BELIEF.parameters(), lr=2e-3)


def train_beliefs(n_games=6000, epochs=200, seed=SEED):
    """Entrainement supervise sur des parties auto-jouees par la politique de base."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    rows, labels = [], []
    for _ in range(n_games):
        _, _, cards, decisions = play_one_game(lambda i: POLICY_BASE[i])
        for iset, _, player in decisions:
            rows.append(encode_infoset(iset))
            labels.append(cards[1 - player])
    x = torch.tensor(np.stack(rows))
    y = torch.tensor(labels)
    for _ in range(epochs):
        b_loss = F.cross_entropy(BELIEF(x), y)
        B_OPTIM.zero_grad()
        b_loss.backward()
        B_OPTIM.step()
    return float(b_loss.detach())


FINAL_CE = train_beliefs()
print(f'Entropie croisee finale du reseau de croyances : {FINAL_CE:.4f}')
print('(reference : ln 2 = 0.693 = hasard pur entre les 2 cartes restantes)')

# La carte du joueur est exclue a priori : on masque avant la normalisation.
with torch.no_grad():
    raw = BELIEF(ENCODED).numpy()
BELIEF_TABLE = {}
for i in ALL_INFOSETS:
    my_card = (i % 6) // 2
    p = raw[i].copy()
    p[my_card] = -1e9
    BELIEF_TABLE[i] = (np.exp(p - p.max()) / np.exp(p - p.max()).sum()).tolist()

print('\nCroyances P(carte adverse | infoset) pour J1 a l ouverture :')
for card in range(3):
    i = infoset_id(0, card, 0)
    row = ', '.join(f'{CARD_NAMES[c]}={BELIEF_TABLE[i][c]:.3f}' for c in range(3))
    print(f'  J1 tient {CARD_NAMES[card]:5s} -> {{{row}}}')
print('\nCroyances sur les infosets de reponse de J2 (J1 vient d encherir) :')
for card in range(3):
    i = infoset_id(1, card, 1)
    row = ', '.join(f'{CARD_NAMES[c]}={BELIEF_TABLE[i][c]:.3f}' for c in range(3))
    print(f'  J2 tient {CARD_NAMES[card]:5s} -> {{{row}}}')

Entropie croisee finale du reseau de croyances : 0.7042
(reference : ln 2 = 0.693 = hasard pur entre les 2 cartes restantes)

Croyances P(carte adverse | infoset) pour J1 a l ouverture :
  J1 tient Valet -> {Valet=0.000, Dame=0.485, Roi=0.515}
  J1 tient Dame  -> {Valet=0.487, Dame=0.000, Roi=0.513}
  J1 tient Roi   -> {Valet=0.507, Dame=0.493, Roi=0.000}

Croyances sur les infosets de reponse de J2 (J1 vient d encherir) :
  J2 tient Valet -> {Valet=0.000, Dame=0.470, Roi=0.530}
  J2 tient Dame  -> {Valet=0.486, Dame=0.000, Roi=0.514}
  J2 tient Roi   -> {Valet=0.507, Dame=0.493, Roi=0.000}


### Interprétation : des croyances à la hauteur de l'information disponible

**À lire sur la sortie ci-dessus.** Pour le joueur 1 à l'ouverture, l'information disponible se réduit à sa propre carte : la distribution exacte de la carte adverse est uniforme sur les **deux** cartes restantes. Le réseau l'apprend (deux poids quasi égaux, un poids quasi nul pour sa propre carte) — sans pouvoir descendre sous le hasard : l'entropie croisée finale vaut **0.7042**, juste **au-dessus** du plancher `ln 2 ≈ 0.693`. Ce n'est pas un échec d'apprentissage, c'est une **conséquence de la politique de base** : elle enchérit 86,6–89,2 % sur les infosets affichés (§3), à peine différenciés selon la carte — et les croyances quasi uniformes ci-dessus en sont la lecture directe : par Bayes, quand P(miser | Valet) ≈ P(miser | Roi), le postérieur reste au prior, et le plancher de l'entropie croisée est exactement `ln 2`. Le réseau de croyances a correctement appris que, face à cette politique, **l'action de l'adversaire n'informe pas**.

Il reste un **résidu mesurable** : sur les infosets de réponse (J1 vient d'enchérir), le réseau qui tient le Valet donne Roi=0.530 contre Dame=0.470 — une inclinaison réelle mais ténue, fidèle aux petits écarts de taux de mise de la politique de base selon la carte tenue et la position. La brique 3 échantillonnera ses mondes depuis ces croyances : ici, elle échantillonnera **presque au hasard** — et c'est honnête, puisque c'est ce que l'observation permet de savoir. L'Exercice 3 demande de confronter ces croyances apprises au conditionnement **exact** calculé par énumération.

## 5. Brique 3 — la recherche au temps de test par *update equivalence*

La brique 3 est le cœur conceptuel du papier : **chercher, c'est appliquer la mise à jour d'entraînement aux croyances du moment**. Concrètement, à l'infoset où le joueur doit agir :

1. **échantillonner** `K` mondes depuis le réseau de croyances (la carte adverse, cohérente avec l'observation) ;
2. pour chaque action candidate, **dérouler** les mondes avec la politique de base (profondeur bornée — sur Kuhn, au plus une décision reste après la nôtre) et **moyenner** les valeurs ;
3. appliquer **un pas tabulaire de descente miroir magnétique** avec les mêmes divergences reverse-KL qu'à l'entraînement : `log pi'(a) = log pi(a) + eta * A(a)`, replié dans une **région de confiance** reverse-KL vers `pi` ;
4. **échantillonner** le coup depuis la politique mise à jour `pi'` — pas argmax.

Le point 4 est ce qui distingue la mise à jour d'équivalence d'une exploitation. Le **témoin négatif** de la §6 construit la variante « recherche **sans** mise à jour » : mêmes mondes, mêmes moyennes de valeurs, mais l'action est l'argmax des valeurs moyennées — la recherche qui **exploite** l'estimation sans jamais réviser sa politique. La comparaison des deux mesure ce que la mise à jour d'équivalence apporte.

In [5]:
def tts_policy(iset, k_samples=64, eta=2.0, kl_delta=0.25, seed=SEED):
    """Recherche au temps de test par update equivalence.
    Rend (pi_mise_a_jour, details) pour l infoset donne."""
    rng = random.Random(seed + iset)
    player, my_card = iset // 6, (iset % 6) // 2
    base = POLICY_BASE[iset]
    belief = BELIEF_TABLE[iset]

    # 1. K mondes echantillonnes depuis le reseau de croyances
    remaining = [c for c in range(3) if c != my_card]
    worlds = rng.choices(remaining, weights=[belief[c] for c in remaining], k=k_samples)
    history = history_of_infoset(iset)

    # 2. valeurs moyennees par action candidate
    def mean_value(action_idx):
        vals = []
        for opp_card in worlds:
            cards = (my_card, opp_card) if player == 0 else (opp_card, my_card)
            h = history + KuhnPoker.legal_actions(history)[action_idx]
            if not KuhnPoker.is_terminal(h):
                vals.append(rollout_return(h, cards, player, lambda i: POLICY_BASE[i]))
            else:
                g = KuhnPoker.payoff(h, cards)
                vals.append(g if player == 0 else -g)
        return float(np.mean(vals))

    v = [mean_value(0), mean_value(1)]
    adv = np.array(v) - np.mean(v)

    # 3. un pas de descente miroir magnetique, region de confiance reverse-KL
    log_base = np.log(np.array(base) + 1e-12)
    target = log_base + eta * adv

    def softmax_np(z):
        e = np.exp(z - z.max())
        return e / e.sum()

    def kl_to_base(p):
        return float(np.sum(p * (np.log(p + 1e-12) - log_base)))

    pi_new = softmax_np(target)
    t_mix = 1.0
    if kl_to_base(pi_new) > kl_delta:  # projection : bisection sur le pas de melange
        lo, hi = 0.0, 1.0
        for _ in range(12):
            mid = 0.5 * (lo + hi)
            p_mid = softmax_np((1 - mid) * log_base + mid * target)
            if kl_to_base(p_mid) <= kl_delta:
                lo = mid
            else:
                hi = mid
        t_mix = lo
        pi_new = softmax_np((1 - t_mix) * log_base + t_mix * target)

    details = {'valeurs': v, 'avantage': adv.tolist(), 't_mix': t_mix,
               'kl_final': kl_to_base(pi_new)}
    return pi_new.tolist(), details


def tts_tabular(k_samples=64, eta=2.0, kl_delta=0.25):
    """Politique TTS complete (12 infosets) : mise a jour d'equivalence."""
    return {i: tts_policy(i, k_samples, eta, kl_delta)[0] for i in ALL_INFOSETS}


def greedy_tabular(k_samples=64):
    """TEMOIN NEGATIF : meme estimation des valeurs, PAS de mise a jour -> argmax deterministe."""
    table = {}
    for i in ALL_INFOSETS:
        _, d = tts_policy(i, k_samples)
        table[i] = [1.0, 0.0] if d['valeurs'][0] >= d['valeurs'][1] else [0.0, 1.0]
    return table


POLICY_TTS = tts_tabular()
POLICY_GREEDY = greedy_tabular()
print('Comparaison sur 3 infosets (base / TTS-update / temoin-argmax) :')
for i in (0, 1, 6):
    print(f'  infoset {i:2d}: base={[round(p, 3) for p in POLICY_BASE[i]]}  '
          f'tts={[round(p, 3) for p in POLICY_TTS[i]]}  greedy={POLICY_GREEDY[i]}')

Comparaison sur 3 infosets (base / TTS-update / temoin-argmax) :
  infoset  0: base=[0.116, 0.884]  tts=[0.11, 0.89]  greedy=[0.0, 1.0]
  infoset  1: base=[0.134, 0.866]  tts=[0.421, 0.579]  greedy=[1.0, 0.0]
  infoset  6: base=[0.108, 0.892]  tts=[0.338, 0.662]  greedy=[1.0, 0.0]


In [6]:
DEALS = [(c1, c2) for c1 in range(3) for c2 in range(3) if c1 != c2]  # 6 donnes uniformes


def expected_value(pi1, pi2):
    """Valeur exacte du jeu pour J1 sous une paire de politiques comportementales,
    par enumeration des 6 donnees."""
    total = 0.0
    for cards in DEALS:
        for h, p in _expand('', cards, pi1, pi2):
            total += p * KuhnPoker.payoff(h, cards) / len(DEALS)
    return total


def _expand(history, cards, pi1, pi2):
    """Enumere (historique terminal, probabilite) sous la paire de politiques."""
    if KuhnPoker.is_terminal(history):
        return [(history, 1.0)]
    player = KuhnPoker.current_player(history)
    pi = pi1 if player == 0 else pi2
    iset = infoset_id(player, cards[player], history_bucket(history))
    out = []
    for a_idx, a in enumerate(KuhnPoker.legal_actions(history)):
        for h2, p2 in _expand(history + a, cards, pi1, pi2):
            out.append((h2, pi[iset][a_idx] * p2))
    return out


def best_response_value(opp_policy, br_player):
    """Valeur de la meilleure reponse EXACTE du joueur br_player contre opp_policy.
    Iteration de politique sur les infosets de br_player (converge en 2 passes sur Kuhn)."""
    br_table = {i: [0.5, 0.5] for i in ALL_INFOSETS if i // 6 == br_player}

    def traverse(history, cards):
        """Valeur (pour br_player) du noeud, br jouant br_table, l adversaire opp_policy."""
        if KuhnPoker.is_terminal(history):
            g = KuhnPoker.payoff(history, cards)
            return g if br_player == 0 else -g
        player = KuhnPoker.current_player(history)
        iset = infoset_id(player, cards[player], history_bucket(history))
        probs = br_table[iset] if player == br_player else opp_policy[iset]
        legal = KuhnPoker.legal_actions(history)
        return sum(probs[a] * traverse(history + legal[a], cards) for a in (0, 1))

    for _ in range(3):  # induction arriere : les decisions profondes d'abord
        for iset in sorted(br_table, key=lambda i: history_bucket(i), reverse=True):
            player, my_card = iset // 6, (iset % 6) // 2
            history = history_of_infoset(iset)
            action_vals = [0.0, 0.0]
            for cards in DEALS:
                if cards[player] != my_card:
                    continue
                legal = KuhnPoker.legal_actions(history)
                for a in (0, 1):
                    action_vals[a] += traverse(history + legal[a], cards) / len(DEALS)
            best = max((0, 1), key=lambda a: action_vals[a])
            br_table[iset] = [1.0, 0.0] if best == 0 else [0.0, 1.0]

    return sum(traverse('', cards) for cards in DEALS) / len(DEALS)


def exploitability(pi1, pi2):
    """Moyenne des gains de meilleure reponse des deux cotes (0 a l equilibre)."""
    br1 = best_response_value(pi2, 0)   # J1 repond au mieux a pi2
    br2 = best_response_value(pi1, 1)   # J2 repond au mieux a pi1
    val = expected_value(pi1, pi2)
    return 0.5 * ((br1 - val) + (br2 + val))


def self_check_exploitability():
    """Controles exacts derives a la main sur la paire uniforme : valeur de Kuhn
    uniforme-vs-uniforme = +1/8 pour J1 (l asymetrie des folds est par ROLE, pas par
    donne : bf rapporte toujours +1 a J1, pbf toujours -1) ; meilleure reponse
    carte-avise contre uniforme : J1 = 1/2, J2 = 5/12, exploitabilite = 11/24."""
    random_pol = {i: [0.5, 0.5] for i in ALL_INFOSETS}
    val = expected_value(random_pol, random_pol)
    expl = exploitability(random_pol, random_pol)
    print(f'controle paire uniforme : valeur = {val:+.6f} (exacte +1/8 = +0.125), '
          f'exploitabilite = {expl:.6f} (exacte 11/24 = 0.458333)')
    assert abs(val - 0.125) < 1e-9, 'valeur de la paire uniforme : +1/8 pour J1'
    assert abs(expl - 11 / 24) < 1e-9, 'exploitabilite de la paire uniforme : 11/24'


self_check_exploitability()
print(f'Reference Nash (cfr carnet 13) : valeur du jeu = {-1/18:+.4f} pour J1')

controle paire uniforme : valeur = +0.125000 (exacte +1/8 = +0.125), exploitabilite = 0.458333 (exacte 11/24 = 0.458333)
Reference Nash (cfr carnet 13) : valeur du jeu = -0.0556 pour J1


In [7]:
def report(policies):
    """Tableau : exploitabilite et valeur pour chaque paire nommee + face-a-face."""
    names = list(policies)
    print(f"{'politique':<20s} {'exploitabilite':>14s} {'valeur J1':>10s}")
    for name, (p1, p2) in policies.items():
        print(f'{name:<20s} {exploitability(p1, p2):>14.4f} '
              f'{expected_value(p1, p2):>+10.4f}')
    print('\nFace-a-face (gain moyen J1 de la 1re politique) :')
    for a in names:
        for b in names:
            if a != b:
                print(f'  {a} (J1) vs {b} (J2) : {expected_value(policies[a][0], policies[b][1]):+.4f}')


POLICIES = {
    'base': (POLICY_BASE, POLICY_BASE),
    'tts-update-equiv': (POLICY_TTS, POLICY_TTS),
    'temoin-argmax': (POLICY_GREEDY, POLICY_GREEDY),
}
report(POLICIES)

politique            exploitabilite  valeur J1
base                         0.3306    +0.0878
tts-update-equiv             0.2653    -0.1152
temoin-argmax                0.1667    -0.3333

Face-a-face (gain moyen J1 de la 1re politique) :
  base (J1) vs tts-update-equiv (J2) : -0.1349
  base (J1) vs temoin-argmax (J2) : -0.3322
  tts-update-equiv (J1) vs base (J2) : +0.1113
  tts-update-equiv (J1) vs temoin-argmax (J2) : -0.3176
  temoin-argmax (J1) vs base (J2) : +0.1092
  temoin-argmax (J1) vs tts-update-equiv (J2) : -0.1314


### Interprétation : ce que la mise à jour d'équivalence apporte — lecture honnête du témoin

**Les trois lignes du tableau se lisent ainsi.** L'exploitabilité mesure la distance d'une paire de politiques à l'exploitabilité nulle : l'écart que la meilleure réponse exacte peut encore arracher. La ligne `base` fixe le point de départ (l'auto-jeu de la brique 1, sans recherche) ; la ligne `tts-update-equiv` mesure la brique 3 complète ; la ligne `temoin-argmax` mesure la **même estimation de valeurs, sans la mise à jour** (l'argmax des valeurs moyennes par action).

**Ce que la mesure dit, et elle renverse le récit attendu.** Les deux variantes améliorent la base (0.331 → 0.265 pour la mise à jour, → 0.167 pour l'argmax), mais c'est le **témoin** qui descend le plus bas ici. La cause tient à la région de confiance : le pas miroir est borné par la KL (budget δ = 0.25) **depuis une base très déséquilibrée**, quand l'argmax saute sans contrainte vers la réponse approchée. Borne le pas, borne la correction : la mise à jour d'équivalence recouvre ici ≈ 40 % du chemin du témoin (0.065 sur 0.164) sans jamais pouvoir s'en écarter de plus de δ. C'est exactement ce qu'une région de confiance est censée faire — **payer la robustesse au prix de la profondeur de correction**.

**Deux axes, pas un.** L'exploitabilité (robustesse face à la meilleure réponse *exacte*) et le face-à-face (force contre un adversaire fixé) ne classent pas pareil : en position de joueur 2, la politique TTS bat **et** la base (+0.135) **et** le témoin argmax (+0.131) ; en position de joueur 1, elle perd contre le témoin (−0.318). Aucune domination stricte — le témoin, déterministe, est fort mais prévisible ; la TTS reste mixte (infoset 1 : [0.42, 0.58]) donc moins modélisable par un adversaire qui apprendrait. Sur ce jouet mono-seed, aucune de ces valeurs n'est un benchmark — elles attestent la **mécanique** : un seul pas de descente miroir tabulaire, guidé par les croyances, déplace la politique dans le bon sens **sans aucun entraînement supplémentaire**, et le témoin sans mise à jour prouve par contraste que c'est bien le pas d'équivalence qui opère le déplacement. Les exercices font varier ce qui déplace ces valeurs.

## 6. Exercices

Trois exercices prolongent la marche. Convention du dépôt : les stubs s'exécutent **sans erreur volontaire** (règle C.1 — le carnet doit tourner de bout en bout même non complété), la fonction renvoie `None` tant que l'exercice n'est pas fait.

### Exercice 1 — la courbe d'exploitabilité contre le nombre de mondes échantillonnés

La brique 3 moyenne des valeurs sur `K` mondes échantillonnés depuis le réseau de croyances. `K` contrôle le compromis **bruit de l'estimateur / coût de la recherche**. Écrivez `mesurer_courbe_exploitabilite(ks)` qui, pour chaque `K` de la liste, reconstruit la politique TTS et son témoin argmax avec ce budget, mesure leur exploitabilité, et rend le dictionnaire `{k: (expl_tts, expl_greedy)}`.

> **Indice :** `tts_tabular(k_samples=k)` et `greedy_tabular(k_samples=k)` acceptent le budget en argument ; `exploitability(pol, pol)` fait la mesure. Question guide : que fait l'exploitabilité du témoin quand `K` grandit — et celle de la mise à jour d'équivalence ?

### Sortie attendue

```
courbe d exploitabilite par K : {4: (...), 8: (...), ...}
```

Le stub imprime le message d'attente et rend `None` tant que l'exercice n'est pas complété.

In [8]:
def mesurer_courbe_exploitabilite(ks=(4, 8, 16, 32, 64, 128)):
    """Exploitabilite des deux variantes de recherche en fonction du budget K de mondes.

    Rend {k: (exploitabilite_tts, exploitabilite_greedy)} ou None si non complete.
    """
    # TODO etudiant : boucler sur ks, construire tts_tabular(k_samples=k) et
    # greedy_tabular(k_samples=k), mesurer exploitability(pol, pol) pour chacun.
    # Indice : tout est deja en scope, rien a importer.
    print('Exercice 1 a completer : courbe d exploitabilite vs budget K de la recherche')
    return None


RESULTAT_EX1 = mesurer_courbe_exploitabilite()

Exercice 1 a completer : courbe d exploitabilite vs budget K de la recherche


### Exercice 2 — inverser l'amortissement dynamique

La §3 coordonne régularisation et pas : forte pénalité reverse-KL et pas agressifs **tôt**, pénalité faible et petits pas **tard**. Écrivez `inverser_amortissement()` qui ré-entraîne un réseau **neuf** avec le calendrier **inversé** (`lambda` faible→fort, `lr` petit→grand), même budget de manches, puis rend `(exploitabilite_inversee, exploitabilite_nominale)` — la seconde en ré-entraînant aussi un réseau neuf avec le calendrier nominal, pour une comparaison à graines égales.

> **Indice :** `train_selfplay(net, optim, lam_start=..., lam_end=..., lr_start=..., lr_end=...)` prend le calendrier en arguments ; un réseau neuf s'obtient par `PolicyValueNet()` avec son propre `torch.optim.Adam`. La méthode `policy_table()` d'un réseau extrait sa politique tabulaire. Question guide : qu'arrive-t-il à la stabilité quand la régularisation se relâche précisément quand la politique cesse d'être rattrapable ?

### Sortie attendue

```
amortissement inverse vs nominal : (expl_inverse, expl_nominale)
```

### Exercice 3 — croyances apprises contre conditionnement exact

Le réseau de croyances apprend `P(carte adverse | infoset)` depuis les données d'auto-jeu. L'exercice calcule le **conditionnement exact** — par énumération des donnes et des trajectoires sous `POLICY_BASE` — et le confronte au réseau. Écrivez `croyances_conditionnees()` qui rend l'écart maximal `max |p_apprise - p_exacte|` sur les 12 infosets.

> **Indice :** pour chaque donne `(c_mienne, c_adv)` (poids `1/6`), le conditionnement exact à l'infoset de réponse pondère la carte adverse par la probabilité que l'adversaire joue l'action observée à **son** infoset sous `POLICY_BASE`. Les infosets d'ouverture (bucket 0) ont un exact uniforme sur les 2 cartes restantes — l'écart y teste seulement l'apprentissage de l'uniforme. `BELIEF_TABLE` porte les probabilités apprises.

### Sortie attendue

```
ecart max croyances apprises vs exactes : <valeur>
```

In [9]:
def inverser_amortissement(n_rounds=28):
    """Re-entraine deux reseaux neufs : calendrier inverse vs nominal, graines egales.

    Rend (exploitabilite_inverse, exploitabilite_nominale) ou None si non complete.
    """
    # TODO etudiant : deux PolicyValueNet() neufs, chacun avec son Adam.
    # Calendrier nominal : lam 0.8 -> 0.05, lr 3e-3 -> 2e-4 (defauts de train_selfplay).
    # Calendrier inverse : lam 0.05 -> 0.8, lr 2e-4 -> 3e-3.
    # Indice : exploitability(pol, pol) pour mesurer chaque reseau.
    print('Exercice 2 a completer : amortissement dynamique inverse')
    return None


RESULTAT_EX2 = inverser_amortissement()


def croyances_conditionnees():
    """Ecart max entre croyances apprises et conditionnement exact, sur 12 infosets.

    Rend max |p_apprise - p_exacte| ou None si non complete.
    """
    # TODO etudiant : pour chaque infoset, calculer p_exacte(c) par enumeration
    # des 6 donnees x trajectoires sous POLICY_BASE, comparer a BELIEF_TABLE[iset].
    # Indice : helper history_of_infoset(iset) rend l historique de l infoset ;
    # POLICY_BASE[iset_adv][a_idx] ponderent l action observee.
    print('Exercice 3 a completer : croyances apprises vs conditionnement exact')
    return None


RESULTAT_EX3 = croyances_conditionnees()

Exercice 2 a completer : amortissement dynamique inverse
Exercice 3 a completer : croyances apprises vs conditionnement exact


## 7. Raccords dans la série, et ce que le grain B ajoutera

| Voisin | Raccord |
|---|---|
| **13** (CFR/MCCFR/Deep CFR) | le socle : le CFR résout par regret **à l'itération**, la brique 1 apprend par gradient **aux données** — deux réponses à la même question ; l'exploitabilité de la §6 les sépare sur le même jouet |
| **13d** (CFR optimiste, OFTRL) | la descente miroir magnétique du temps de test est de la **même famille de mise à jour** que l'OFTRL — le 13d est l'ancre théorique de la brique 3 |
| **13e** (révision AGM des croyances) | le réseau de croyances fait le **même geste en appris** : 13e révisait symboliquement, ici la distribution s'apprend — témoin négatif réciproque |
| **19** (abstraction à dette) | le réseau de croyances est une **compression de l'information cachée** — frontière à tenir à l'œil quand le jouet grandit |

**Ce que le grain B ajoutera** : l'organe natif [`AtaraxosAI/doudizhu`](https://github.com/AtaraxosAI/doudizhu) (MIT, Python) invoqué sur un budget réduit — installer, exécuter, et rapporter **ce qui ne survit pas** à la réduction d'échelle. Ce carnet démontre le patron sur un jouet ; le grain B confronte le patron au vrai moteur.

**Transition dans la série** : le 13f se lit après le 13 (le CFR fournit le point de comparaison) et en parallèle du 13e (les croyances). Les exercices 1-3 couvrent les trois degrés de liberté du patron — budget de recherche, calendrier d'amortissement, qualité des croyances — qui sont exactement ceux que le papier règle à l'échelle.